In [ ]:
import pandas as pd

In [ ]:
import pandas as pd
import numpy as np


def parse_duration(value):
    """
    Examples:
        11:55:00 -> 11 min 55 sec
        5:04     -> 5 min 4 sec
        :20      -> 20 sec

    Returns:
        pd.Timedelta
    """

    if pd.isna(value):
        return pd.Timedelta(seconds=0)

    value = str(value).strip()

    if value == "":
        return pd.Timedelta(seconds=0)

    # :20
    if value.startswith(":"):
        return pd.Timedelta(seconds=int(value[1:]))

    parts = value.split(":")

    # 11:55:00
    if len(parts) == 3:
        return pd.Timedelta(
            minutes=int(parts[0]),
            seconds=int(parts[1])
        )

    # 5:04
    if len(parts) == 2:
        return pd.Timedelta(
            minutes=int(parts[0]),
            seconds=int(parts[1])
        )

    return pd.Timedelta(seconds=0)


# ==================================================
# Read File
# ==================================================

file = r"C:\Users\mturky\Documents\Copy of rep2.xlsx"

raw = pd.read_excel(file, header=None)

headers = raw.iloc[3]

df = raw.iloc[4:].copy()
df.columns = headers

# ==================================================
# Clean Data
# ==================================================

df = df[df["Time"] != "Totals"]

df.columns.values[2] = "To"

df = df.drop(df.columns[1], axis=1)

df = df.drop(
    columns=["Flow In", "Flow Out"],
    errors="ignore"
)

df = df.reset_index(drop=True)

# ==================================================
# Time Columns
# ==================================================

df["To"] = pd.to_datetime(
    df["To"],
    format="%I:%M%p"
)

df["Time"] = (
    df["To"] -
    pd.Timedelta(minutes=30)
)

df["To"] = df["To"].dt.strftime("%I:%M %p")
df["Time"] = df["Time"].dt.strftime("%I:%M %p")

# ==================================================
# Numeric Columns
# ==================================================

df["ACD Calls"] = pd.to_numeric(
    df["ACD Calls"],
    errors="coerce"
).fillna(0)

df["Aban Calls"] = pd.to_numeric(
    df["Aban Calls"],
    errors="coerce"
).fillna(0)

df["AVG STAFF"] = pd.to_numeric(
    df["AVG STAFF"],
    errors="coerce"
).fillna(0)

# ==================================================
# Duration Column
# ==================================================

df["Avg Speed Ans"] = (
    df["Avg Speed Ans"]
    .apply(parse_duration)
)

# ==================================================
# Save Detailed Data
# ==================================================

df.to_csv(
    "bcms_temp.csv",
    index=False
)

# ==================================================
# Summary
# ==================================================

# summary = (
#     df.groupby("Split/Skill")
#       .agg(
#           ACD_Calls=("ACD Calls", "sum"),
#           Aban_Calls=("Aban Calls", "sum"),
#           Total_Speed_Ans=("Avg Speed Ans", "sum"),
#           AVG_Speed_Ans=("Avg Speed Ans", "mean"),
#           Avg_Staff=("AVG STAFF", "mean")
#       )
#       .reset_index()
# )


summary = (
    df.groupby("Split/Skill")
      .agg(
          ACD_Calls=("ACD Calls", "sum"),
          Aban_Calls=("Aban Calls", "sum"),
          Total_Speed_Ans=("Avg Speed Ans", "sum"),
          Avg_Speed_Ans=(
              "Avg Speed Ans",
              lambda x: x[df.loc[x.index, "ACD Calls"] > 0].mean()
          ),
          Avg_Staff=("AVG STAFF", "mean")
      )
      .reset_index()
)

summary.to_csv('bcms_summary1.csv', index = False)

# Offered Calls

summary["Offered Calls"] = (
    summary["ACD_Calls"] +
    summary["Aban_Calls"]
)

# Abandon Rate %

summary["Abandon Rate %"] = (
    summary["Aban_Calls"]
    .div(summary["Offered Calls"].replace(0, np.nan))
    .fillna(0)
    .mul(100)
    .round(2)
)

# Calls per Agent

summary["Calls per Agent"] = (
    summary["ACD_Calls"]
    .div(summary["Avg_Staff"].replace(0, np.nan))
    .fillna(0)
    .round(2)
)



summary["Avg_Staff"] = (
    summary["Avg_Staff"]
    .round(2)
)

# Remove helper column

summary = summary.drop(
    columns=["Total_Speed_Ans"]
)

# Sort

summary = summary.sort_values(
    "Abandon Rate %",
    ascending=False
)

# Display

# print(summary)

# Export

print(summary)

summary["Avg_Speed_Ans"] = (
    summary["Avg_Speed_Ans"]
    .astype(str)
    .str.replace("0 days ", "", regex=False)
    .str.split(".")
    .str[0]
    .fillna("00:00:00")
    .replace("NaT", "00:00:00")
)

summary['Avg_Speed_Ans_seconds'] =  pd.to_timedelta(summary["Avg_Speed_Ans"]).dt.total_seconds()

summary.to_csv(
    "bcms_summary.csv",
    index=False
)




peak = (
    df.groupby("Time")["ACD Calls"]
      .sum()
    #   .sort_values(ascending=False)
)
# display (peak)
peak = df.groupby(['Time','To']).agg(ACD_Calls = ("ACD Calls",'sum'),Aban_Calls=('Aban Calls','sum')).reset_index().sort_values('ACD_Calls',ascending = False)


peak.to_csv('bcms_peak.csv', index = False)

In [ ]:
df.info()